In [1]:
from torch.utils.data import Dataset
import pandas as pd

class PlotDataset(Dataset):
    def __init__(self, df, num_species, source_filter=None, subset=None):
        if source_filter is not None:
            df = df[df["source"] == source_filter]
        if subset is not None:
            df = df[df["subset"] == subset]

        # parse once: strings like "(1, 2, 3)" → set of ints
        df = df.copy()
        df["species_set"] = (
            df["species_set"]
            .str.strip("{}")              # remove surrounding braces
            .str.split(",")               # split by comma
            .apply(lambda xs: {int(x) for x in xs if x.strip()})
        )

        self.df = df.reset_index(drop=True)
        self.num_species = num_species

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = torch.zeros(self.num_species, dtype=torch.float32)
        if row["species_set"]:
            x[list(row["species_set"])] = 1.0
        return {"survey_id": row["survey_id"], "x": x}

In [2]:
from torch.utils.data import DataLoader


COUNTRY = ['DK']
NUM_SPECIES = 15286

# --- Train metadata ---
train_metadata = pd.read_csv(f"./metadata/GPNv2-{COUNTRY}-metadata_train.csv")

pa_train_ds = PlotDataset(train_metadata, NUM_SPECIES, source_filter="PA", subset="train")
po_train_ds = PlotDataset(train_metadata, NUM_SPECIES, source_filter="PO", subset="train")

pa_train_loader = DataLoader(pa_train_ds, batch_size=64, shuffle=True, num_workers=10)
po_train_loader = DataLoader(po_train_ds, batch_size=64, shuffle=True, num_workers=10)

print("PA train size:", len(pa_train_ds))
print("PO train size:", len(po_train_ds))

# --- Val metadata ---
val_metadata = pd.read_csv(f"./metadata/GPNv2-{COUNTRY}-metadata_val.csv")

pa_val_ds = PlotDataset(val_metadata, NUM_SPECIES, source_filter="PA", subset="val")
po_val_ds = PlotDataset(val_metadata, NUM_SPECIES, source_filter="PO", subset="val")

pa_val_loader = DataLoader(pa_val_ds, batch_size=64, shuffle=False, num_workers=10)
po_val_loader = DataLoader(po_val_ds, batch_size=64, shuffle=False, num_workers=10)

print("PA val size:", len(pa_val_ds))
print("PO val size:", len(po_val_ds))

PA train size: 187586
PO train size: 496781
PA val size: 1318
PO val size: 2701


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerSpecies(nn.Module):
    def __init__(self, num_species: int, d: int = 256, hidden: int = 1024, dropout=0.1):
        super().__init__()
        self.num_species = num_species
        # Plot tower (multi-hot -> embedding)
        self.plot_mlp = nn.Sequential(
            nn.Linear(num_species, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d),
            nn.ReLU(),
        )
        # Species tower (embedding table + bias)
        self.spec_emb = nn.Embedding(num_species, d)
        self.spec_bias = nn.Embedding(num_species, 1)
        nn.init.xavier_uniform_(self.spec_emb.weight)
        nn.init.zeros_(self.spec_bias.weight)

    def plot_encode(self, x):
        return self.plot_mlp(x)

    def score_indices(self, z, idx):
        # z: [B,d] (maybe bf16 under autocast)
        # gather embeddings in fp32, do matmul in fp32 for stability
        z32 = z.float()
        E32 = self.spec_emb(idx).float()              # [K,d]
        b32 = self.spec_bias(idx).squeeze(1).float()  # [K]
        logits32 = z32 @ E32.t() + b32                # [B,K]
        return logits32.to(z.dtype)                   # back to original dtype

    def forward_full(self, x):
        z = self.plot_encode(x)                       # [B,d] maybe bf16
        z32 = z.float()
        W32 = self.spec_emb.weight.float()            # [S,d]
        b32 = self.spec_bias.weight.squeeze(1).float()# [S]
        logits32 = z32 @ W32.t() + b32                # [B,S]
        return logits32.to(z.dtype)

In [4]:
def get_batch_pos_indices(x):
    # union of positive columns in the batch
    return torch.where(x.sum(0) > 0)[0]

def build_targets_from_indices(x, idx):
    return x.index_select(1, idx).float()

def sample_negatives(exclude_idx, S, K):
    mask = torch.ones(S, dtype=torch.bool, device=exclude_idx.device)
    mask[exclude_idx] = False
    pool = torch.where(mask)[0]
    K = min(K, pool.numel())
    perm = torch.randperm(pool.numel(), device=pool.device)[:K]
    return pool[perm]

In [5]:
# ------- PA: weighted BCE on positives + sampled true zeros -------
def loss_pa_weighted_bce(logits, targets, pos_weight=5.0):
    # logits/targets: [B,K]
    # pos_weight -> scale positives to counter class imbalance
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=torch.tensor(pos_weight, device=logits.device))

# ------- PO: BPR pairwise ranking loss (z·e_pos > z·e_neg) -------
def loss_bpr(pos_scores, neg_scores):  # both [B, M]
    return -torch.mean(F.logsigmoid(pos_scores - neg_scores))

In [6]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

def _skip_constant_cols(y_true: np.ndarray, y_score: np.ndarray):
    """Remove columns where targets are all 0 or all 1 to make AUC valid."""
    keep = ~(np.all(y_true == 0, axis=0) | np.all(y_true == 1, axis=0))
    return y_true[:, keep], y_score[:, keep]

@torch.no_grad()
def evaluate_pa_sampled_auc(model,
                            loader,
                            device,
                            num_species,
                            ks=(5, 10, 20),
                            neg_per_pos=50,
                            maxK=4096):
    model.eval()

    # ---------- PASS 1: collect global positives ----------
    pos_global = torch.zeros(num_species, dtype=torch.bool, device=device)
    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)  # [B, S]
        pos_global |= (x.sum(0) > 0)

    pos_idx = torch.where(pos_global)[0]              # [P]
    # sample a fixed set of negatives for the eval epoch
    K_neg = min(maxK, int(neg_per_pos * max(1, pos_idx.numel())))
    neg_idx = sample_negatives(pos_idx, num_species, K_neg)  # function from earlier
    # assemble fixed eval index set (cap to maxK total if needed)
    idx_eval = torch.cat([pos_idx, neg_idx], dim=0)
    if idx_eval.numel() > maxK:
        idx_eval = idx_eval[:maxK]
    K = idx_eval.numel()

    # ---------- PASS 2: evaluate on fixed idx_eval ----------
    bce_sum, n_batches = 0.0, 0
    rec_counts = {k: 0.0 for k in ks}
    n_plots = 0

    all_targets = []
    all_preds = []

    loop = tqdm(loader, leave=False, desc=f"Eval PA (sampled K={K})")
    for batch in loop:
        x = batch["x"].to(device, non_blocking=True)        # [B, S]
        B = x.size(0)

        # slice to fixed columns
        targets = x.index_select(1, idx_eval)               # [B, K]

        # forward
        z = model.plot_encode(x)
        logits = model.score_indices(z, idx_eval)           # [B, K]
        probs = torch.sigmoid(logits)

        # BCE on this reduced set
        loss = F.binary_cross_entropy_with_logits(logits, targets.float())
        bce_sum += loss.item(); n_batches += 1
        loop.set_postfix(bce=f"{loss.item():.4f}")

        # stash for global AUC
        all_targets.append(targets.detach().cpu().numpy())
        all_preds.append(probs.detach().cpu().numpy())

        # Recall@k over reduced set
        n_plots += B
        topk_idx = torch.topk(probs, k=max(ks), dim=1).indices  # [B, maxK]
        for k in ks:
            idx_k = topk_idx[:, :k]
            hits = targets.gather(1, idx_k).sum(dim=1)          # [B]
            denom = targets.sum(dim=1).clamp_min(1)
            rec_counts[k] += (hits / denom).sum().item()

    # concatenate and compute AUC with constant-column filtering
    y_true = np.concatenate(all_targets, axis=0)   # [N, K]
    y_score = np.concatenate(all_preds, axis=0)    # [N, K]
    yt_f, ys_f = _skip_constant_cols(y_true, y_score)
    auc_macro = roc_auc_score(yt_f, ys_f, average='macro') if yt_f.size else float('nan')

    results = {
        "BCE": bce_sum / max(1, n_batches),
        "AUC_macro": auc_macro,
    }
    for k in ks:
        results[f"Recall@{k}"] = rec_counts[k] / max(1, n_plots)
    return results

In [7]:
from tqdm import tqdm

def step_pa(model, batch, opt, device, neg_per_pos=64, drop_prob=0.8, pos_weight=5.0, maxK=4096):
    model.train()
    x = batch["x"].to(device, non_blocking=True)             # true PA labels
    # input corruption: drop presences to mimic PO
    x_in = x.clone()
    drop_mask = (torch.rand_like(x_in) < drop_prob) & (x_in == 1)
    x_in[drop_mask] = 0.0

    # indices: all positives in batch + sampled negatives
    pos_idx = get_batch_pos_indices(x)                       # [P]
    K_neg = min(maxK, int(neg_per_pos * max(1, pos_idx.numel())))
    neg_idx = sample_negatives(pos_idx, model.num_species, K_neg)  # [N]
    idx = torch.cat([pos_idx, neg_idx], dim=0)               # [K]
    targets = build_targets_from_indices(x, idx)             # [B,K]

    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        z = model.plot_encode(x_in)                          # [B,d]
        logits = model.score_indices(z, idx)                 # [B,K]
        loss = loss_pa_weighted_bce(logits, targets, pos_weight=pos_weight)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    return loss.item()

def step_po_bpr(model, batch, opt, device, neg_per_pos=20, max_pos_per_plot=20):
    model.train()
    x = batch["x"].to(device)
    B, S = x.shape

    # encode once
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        z = model.plot_encode(x)  # [B,d]

    # collect positives
    pos_idx = [torch.where(x[b] > 0)[0] for b in range(B)]
    if not any(len(p) for p in pos_idx):
        return 0.0

    total_loss = 0.0; n_terms = 0
    for b in range(B):
        if pos_idx[b].numel() == 0:
            continue
        # subsample positives
        if pos_idx[b].numel() > max_pos_per_plot:
            pos_idx[b] = pos_idx[b][torch.randperm(pos_idx[b].numel(), device=x.device)[:max_pos_per_plot]]
        # sample negatives
        neg_idx = sample_negatives(pos_idx[b], model.num_species, neg_per_pos * pos_idx[b].numel())

        # scores
        pos_s = model.score_indices(z[b:b+1], pos_idx[b]).squeeze(0)  # [P]
        neg_s = model.score_indices(z[b:b+1], neg_idx).squeeze(0)     # [N]

        # BPR loss
        diff = pos_s.unsqueeze(1) - neg_s.unsqueeze(0)                # [P,N]
        loss = -F.logsigmoid(diff).mean()

        total_loss += loss
        n_terms += 1

    if n_terms > 0:
        total_loss = total_loss / n_terms
        opt.zero_grad(set_to_none=True)
        total_loss.backward()
        opt.step()
        return total_loss.item()
    return 0.0

In [8]:
import torch, os

def save_checkpoint(path, model, optimizer=None, epoch=None, metrics=None, calib=None, cfg=None):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    payload = {
        "state_dict": model.state_dict(),
        "epoch": epoch,
        "metrics": metrics or {},       # e.g. {"R@20": 0.67, "BCE": 0.012}
        "calibration": calib or {},     # e.g. {"T": 1.3, "c": 2.1, "tau": 0.5}
        "cfg": cfg or {},               # hyperparams you want to remember
    }
    if optimizer is not None:
        payload["optimizer"] = optimizer.state_dict()
    torch.save(payload, path)
    print(f"✅ Saved checkpoint to {path}")

def load_checkpoint(path, model, optimizer=None, device="mps"):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["state_dict"], strict=True)
    if optimizer is not None and "optimizer" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer"])
    print(f"✅ Loaded model from {path} (epoch={ckpt.get('epoch')})")
    return ckpt  # returns dict so you can read metrics/calibration

In [9]:
import torch
device = torch.device("mps")

model = TwoTowerSpecies(NUM_SPECIES, d=256, hidden=1024, dropout=0.1).to(device)
model = torch.compile(model)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)

# ---- Stage 1: PA pretrain ----
for epoch in range(25):
    pbar = tqdm(pa_train_loader, desc=f"PA epoch {epoch+1}")
    losses = []
    for batch in pbar:
        loss = step_pa(model, batch, opt, device, drop_prob=0.8, neg_per_pos=64, pos_weight=5.0, maxK=4096)
        losses.append(loss); pbar.set_postfix(loss=f"{loss:.4f}")
    val = evaluate_pa_sampled_auc(model, pa_val_loader, device, NUM_SPECIES, ks=(5,10,20))
    print(f"[PA] e{epoch+1} | train {sum(losses)/len(losses):.4f} | "
          f"BCE {val['BCE']:.4f} | AUCm {val['AUC_macro']:.4f} "
          f"R@5 {val['Recall@5']:.3f} | R@10 {val['Recall@10']:.3f} | R@20 {val['Recall@20']:.3f}")

PA epoch 1:   0%|          | 0/2932 [00:00<?, ?it/s]Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'PlotDataset' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
PA epoch 1:   0%|          | 0/2932 [06:55<?, ?it/s]


KeyboardInterrupt: 

In [13]:
# ---- Stage 2: PO finetune ----
for epoch in range(25):
    pbar = tqdm(po_train_loader, desc=f"PO epoch {epoch+1}")
    losses = []
    for batch in pbar:
        loss = step_po_bpr(model, batch, opt, device, neg_per_pos=20, max_pos_per_plot=20)
        losses.append(loss); pbar.set_postfix(loss=f"{loss:.4f}")
    val = evaluate_pa_sampled_auc(model, pa_val_loader, device, NUM_SPECIES, ks=(5,10,20))
    print(f"[PO] e{epoch+1} | train {sum(losses)/len(losses):.4f} | "
          f"BCE {val['BCE']:.4f} | AUCm {val['AUC_macro']:.4f} "
          f"R@5 {val['Recall@5']:.3f} | R@10 {val['Recall@10']:.3f} | R@20 {val['Recall@20']:.3f}")

PO epoch 1: 100%|██████████| 7763/7763 [06:42<00:00, 19.30it/s, loss=0.0016]


[PO] e1 | train 0.0041 | BCE 0.0351 | AUCm 0.9790 R@5 0.361 | R@10 0.616 | R@20 0.834


PO epoch 2: 100%|██████████| 7763/7763 [06:40<00:00, 19.36it/s, loss=0.0009]


[PO] e2 | train 0.0010 | BCE 0.0549 | AUCm 0.9802 R@5 0.364 | R@10 0.628 | R@20 0.853


PO epoch 3: 100%|██████████| 7763/7763 [06:40<00:00, 19.39it/s, loss=0.0000]


...

[PO] e23 | train 0.0002 | BCE 0.1413 | AUCm 0.9950 R@5 0.367 | R@10 0.641 | R@20 0.885


PO epoch 24: 100%|██████████| 7763/7763 [06:36<00:00, 19.59it/s, loss=0.0000]


[PO] e24 | train 0.0002 | BCE 0.2753 | AUCm 0.9945 R@5 0.366 | R@10 0.640 | R@20 0.885


PO epoch 25: 100%|██████████| 7763/7763 [06:42<00:00, 19.27it/s, loss=0.0007]


[PO] e25 | train 0.0002 | BCE 0.4331 | AUCm 0.9940 R@5 0.366 | R@10 0.639 | R@20 0.884


In [14]:
torch.save(model.state_dict(), f"twotower_v0.1-no-calibration.pt")

In [1]:
save_checkpoint(path=f"twotower_v0.1-no-calibration.pt",
            model=model,
            optimizer=optimizer,
            epoch=epoch,
            metrics=val,
            calib=None,  # you can fill this after calibration
            cfg={"num_species": NUM_SPECIES, "d": 256, "hidden": 1024}
        )

NameError: name 'save_checkpoint' is not defined

In [34]:
def init_species_bias_from_pa(model, pa_loader, device="mps", eps=1e-6):
    model.eval()
    S = model.num_species
    pos = torch.zeros(S, dtype=torch.float64, device=device)
    cnt = torch.zeros(S, dtype=torch.float64, device=device)
    with torch.no_grad():
        for batch in pa_loader:
            x = batch["x"].to(device, non_blocking=True).float()  # [B,S]
            pos += x.sum(0)
            cnt += x.shape[0]
    prev = (pos / cnt).clamp(min=eps, max=1-eps)                  # prevalence ∈ (0,1)
    prior_logit = torch.log(prev/(1-prev))                        # logit prevalence
    with torch.no_grad():
        model.spec_bias.weight[:, 0].copy_(prior_logit.float())

In [52]:
import torch
import numpy as np

def _parse_species_set(x):
    if isinstance(x, (set, list, tuple)):
        return set(int(i) for i in x)
    if isinstance(x, str):
        s = x.strip("{}")
        if not s:
            return set()
        return set(int(t.strip()) for t in s.split(",") if t.strip())
    return set()


@torch.no_grad()
def eval_single_plot_drop_trials_simple(
    model, pa_row, num_species,
    device="mps", trials=10, drop_prob=0.8, ks=(5,10,20),
    tau=None, use_bf16=True, seed=None, print_summary=True,
    T: float = 1.0, bias_c: float = 0.0   # <— NEW
):
    """Evaluate one PA plot with repeated random drops, aggregate mean±std."""
    # --- true vector ---
    true_ids = _parse_species_set(pa_row["species_set"])
    x_true = torch.zeros(1, num_species, dtype=torch.float32, device=device)
    if true_ids:
        x_true[0, list(true_ids)] = 1.0
    true_rich = int(x_true.sum().item())

    ctx = (
        torch.autocast(device_type="mps", dtype=torch.bfloat16)
        if (isinstance(device, str) and device.startswith("mps") and use_bf16)
        else torch.cpu.amp.autocast(enabled=False)
    )
    rng = np.random.default_rng(seed)

    per_trial = []
    max_k = max(ks)

    for _ in range(trials):
        # --- create PO-like input by dropping ---
        x_in = x_true.clone()
        if true_rich > 0:
            ones = torch.where(x_true[0] > 0.5)[0].cpu().numpy()
            n_drop = int(np.floor(drop_prob * ones.size))
            if n_drop > 0:
                drop_idx = torch.as_tensor(
                    rng.choice(ones, size=n_drop, replace=False),
                    device=device,
                    dtype=torch.long,
                )
                x_in[0, drop_idx] = 0.0

        # --- forward ---
        with ctx:
            logits = model.forward_full(x_in)           # [1,S]
            # apply calibration
            logits = logits / T + bias_c                # <— NEW
            probs  = torch.sigmoid(logits)
            
        # --- metrics ---
        trial = {}
        top_idx = torch.topk(probs, k=max_k, dim=1).indices
        denom = max(1, true_rich)

        for k in ks:
            hits = int(x_true.gather(1, top_idx[:, :k]).sum().item())
            trial[f"R@{k}"] = hits / denom
            trial[f"P@{k}"] = hits / k

        trial["BCE"] = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, x_true
        ).item()
        trial["true_rich"] = true_rich
        trial["input_rich"] = int(x_in.sum().item())

        # --- new: sums ---
        trial["sum_logits"] = float(logits.sum().item())
        trial["sum_probs"] = float(probs.sum().item())

        if tau is not None:
            pred_bin = (probs >= tau).float()
            tp = int((pred_bin * x_true).sum().item())
            fp = int(((pred_bin == 1) & (x_true == 0)).sum().item())
            fn = int(((pred_bin == 0) & (x_true == 1)).sum().item())
            prec = tp / max(1, tp + fp)
            rec = tp / max(1, tp + fn)
            f1 = (2 * prec * rec) / max(1e-8, (prec + rec))
            trial["F1@tau"] = f1
            trial["tau"] = float(tau)
            trial["pred_rich@tau"] = int(pred_bin.sum().item())

        per_trial.append(trial)

    # --- summarize ---
    def _agg(key):
        vals = [t[key] for t in per_trial if key in t]
        return {
            "mean": float(np.mean(vals)),
            "std": float(np.std(vals)),
            "min": float(np.min(vals)),
            "max": float(np.max(vals)),
        } if vals else None

    keys = [f"R@{k}" for k in ks] + [f"P@{k}" for k in ks] + [
        "BCE", "true_rich", "input_rich", "sum_logits", "sum_probs"
    ]
    if tau is not None:
        keys += ["F1@tau", "pred_rich@tau", "tau"]

    summary = {k: _agg(k) for k in keys if _agg(k) is not None}

    # --- print ---
    if print_summary:
        sid = pa_row.get("survey_id", "<plot>")
        hdr = f"Plot {sid} | trials={trials} | drop={drop_prob:.2f} | true_rich={true_rich}"
        print(hdr)
        def fmt(stat): return f"{stat['mean']:.4f}±{stat['std']:.4f} (min {stat['min']:.4f}, max {stat['max']:.4f})"
        for k in ks:
            print(f"  R@{k}: {fmt(summary[f'R@{k}'])}   P@{k}: {fmt(summary[f'P@{k}'])}")
        print(f"  BCE : {fmt(summary['BCE'])}")
        print(f"  input_rich: {fmt(summary['input_rich'])}")
        print(f"  sum_logits: {fmt(summary['sum_logits'])}")
        print(f"  sum_probs : {fmt(summary['sum_probs'])}")
        if tau is not None and "F1@tau" in summary:
            print(f"  F1@tau: {fmt(summary['F1@tau'])}   pred_rich@tau: {fmt(summary['pred_rich@tau'])}   tau: {fmt(summary['tau'])}")

    return per_trial, summary

In [53]:
test_metadata = pd.read_csv(f"../metadata/GPNv2-{COUNTRY}-metadata_test.csv")
test_metadata.head(5)

,country,latitude,longitude,year,eunis_habitat,x_y,survey_id,species_id,PA,species_set
0,DK,7.694222e+06,1.024074e+06,2017,NaN,50981_71884,2017_50981_71884,4170,True,"{2192, 1298, 1301, 2601, 2857, 810, 428, 2349,..."
1,DK,7.624971e+06,1.014874e+06,2019,NaN,50797_70499,2019_50797_70499,464,True,"{4, 391, 5255, 2192, 9360, 1298, 2079, 2080, 2..."
2,DK,7.355219e+06,1.110116e+06,2018,NaN,52702_65104,2018_52702_65104,3886,True,"{4, 4752, 4247, 666, 2079, 1577, 3886, 2737, 2..."
3,DK,7.725959e+06,1.071108e+06,2019,NaN,51922_72519,2019_51922_72519,3886,True,"{3968, 1412, 1669, 2052, 4, 1672, 4228, 2698, ..."
4,DK,7.452738e+06,1.245502e+06,2015,NaN,55410_67054,2015_55410_67054,3886,True,"{2181, 639, 8977, 2449, 4628, 6040, 2974, 31, ..."


In [50]:
row = pick_pa_plot(test_metadata, idx=1)
per_trial, summary = eval_single_plot_drop_trials_simple(
    model, row, NUM_SPECIES,
    device=device, trials=10, drop_prob=0.8, ks=(5,10,20),
    tau=0.5,  # or None
    seed=42
)

Plot 2019_50797_70499 | trials=10 | drop=0.80 | true_rich=36
  R@5: 0.1389±0.0000 (min 0.1389, max 0.1389)   P@5: 1.0000±0.0000 (min 1.0000, max 1.0000)
  R@10: 0.2583±0.0178 (min 0.2222, max 0.2778)   P@10: 0.9300±0.0640 (min 0.8000, max 1.0000)
  R@20: 0.4333±0.0377 (min 0.3611, max 0.5000)   P@20: 0.7800±0.0678 (min 0.6500, max 0.9000)
  BCE : 0.0233±0.0030 (min 0.0196, max 0.0296)
  input_rich: 8.0000±0.0000 (min 8.0000, max 8.0000)
  sum_logits: -493977.6000±35623.4279 (min -552960.0000, max -452608.0000)
  sum_probs : 0.1066±0.0684 (min 0.0133, max 0.2256)
  F1@tau: 0.0000±0.0000 (min 0.0000, max 0.0000)   pred_rich@tau: 0.0000±0.0000 (min 0.0000, max 0.0000)   tau: 0.5000±0.0000 (min 0.5000, max 0.5000)


In [1]:
import torch, numpy as np
from tqdm import tqdm

@torch.no_grad()
def calibrate_temperature_and_bias(
    model,
    loader,
    device="mps",
    T_grid=np.linspace(0.5, 3.0, 11),
    c_grid=np.linspace(-4.0, 4.0, 17),
):
    """
    Find (T, c) so that mean predicted richness at tau=0.5 matches PA mean richness.
    Shows a progress bar over the grid search.
    """
    # 1) true mean richness
    tot_true, n_plots = 0, 0
    for batch in loader:
        x = batch["x"].to(device)
        tot_true += x.sum().item()
        n_plots += x.size(0)
    true_mean = tot_true / n_plots

    # 2) grid search with progress bar
    best = {"gap": 1e9, "T": 1.0, "c": 0.0, "pred_mean": None}
    total = len(T_grid) * len(c_grid)

    for T in tqdm(T_grid, desc="Calibrating T,c grid", leave=True):
        for c in c_grid:
            tot_pred, n_plots = 0, 0
            for batch in loader:
                x = batch["x"].to(device)
                logits = model.forward_full(x)              # [B,S]
                logits_cal = logits / T + c
                probs = torch.sigmoid(logits_cal)
                pred = (probs >= 0.5).float()
                tot_pred += pred.sum().item()
                n_plots += x.size(0)
            pred_mean = tot_pred / n_plots
            gap = abs(pred_mean - true_mean)
            if gap < best["gap"]:
                best = {
                    "gap": gap,
                    "T": float(T),
                    "c": float(c),
                    "pred_mean": pred_mean,
                }

    print(f"[CAL] true_mean={true_mean:.2f} | pred_mean={best['pred_mean']:.2f} | "
          f"T={best['T']:.3f} c={best['c']:.3f} | gap={best['gap']:.2f}")
    return best["T"], best["c"]

In [ ]:
# Calibrate once (use your PA val loader)
T, c = calibrate_temperature_and_bias(model, pa_val_loader, device)

# Now your per-plot sanity check with calibrated logits
per_trial, summary = eval_single_plot_drop_trials_simple(
    model, row, NUM_SPECIES, device=device, trials=10, drop_prob=0.8,
    ks=(5,10,20), tau=0.5, seed=42, T=T, bias_c=c
)